# Stock Price Prediction using LSTM

A time-series forecasting project that uses a stacked Long Short-Term Memory (LSTM) neural network to model historical AAPL closing prices and generate short-term forecasts.

## Workflow

Historical AAPL data → preprocessing → Min-Max scaling → train/test split → 100-step sequences → stacked LSTM → prediction → RMSE evaluation → 30-step future forecast.

## Model

- 3 stacked LSTM layers
- 50 units per LSTM layer
- 1 dense output unit
- Adam optimizer
- Mean squared error loss
- 100 epochs
- Batch size: 64

## Dataset

The notebook expects `AAPL.csv` with a `close` column.

Place the file at:

`data/AAPL.csv`

The notebook also contains a GitHub Raw URL fallback so it can be opened from this repository in Google Colab once the dataset has been committed.

In [ ]:
# ============================================================
# Configuration
# ============================================================

from pathlib import Path

DATASET_NAME = "AAPL.csv"
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/"
    "saurabh551/Stock-Price-Prediction-LSTM/main/data/AAPL.csv"
)

TIME_STEP = 100
TRAIN_RATIO = 0.65
FORECAST_DAYS = 30

EPOCHS = 100
BATCH_SIZE = 64
RANDOM_STATE = 42

In [ ]:
# ============================================================
# Imports and reproducibility
# ============================================================

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)

In [ ]:
# ============================================================
# Load dataset
# ============================================================

local_candidates = [
    Path("data") / DATASET_NAME,
    Path("..") / "data" / DATASET_NAME,
    Path("/content/data") / DATASET_NAME,
    Path("/content") / DATASET_NAME,
    Path("/content/sample_data") / DATASET_NAME,
]

df = None
loaded_from = None

for candidate in local_candidates:
    if candidate.exists():
        df = pd.read_csv(candidate)
        loaded_from = str(candidate)
        break

if df is None:
    try:
        df = pd.read_csv(GITHUB_RAW_URL)
        loaded_from = GITHUB_RAW_URL
    except Exception as exc:
        raise FileNotFoundError(
            "AAPL.csv was not found locally and could not be loaded from GitHub. "
            "Place AAPL.csv in data/ or upload it to the repository."
        ) from exc

if "close" not in df.columns:
    raise ValueError(
        f"The dataset must contain a 'close' column. Found: {list(df.columns)}"
    )

print("Loaded from:", loaded_from)
print("Rows:", len(df))
print("Columns:", list(df.columns))

display(df.head())

In [ ]:
# ============================================================
# Basic cleaning and target selection
# ============================================================

df = df.copy()

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)

df["close"] = pd.to_numeric(df["close"], errors="coerce")
df = df.dropna(subset=["close"]).reset_index(drop=True)

close_prices = df["close"].to_numpy(dtype=np.float32).reshape(-1, 1)

print("Clean rows:", len(df))
print("Start:", df["date"].iloc[0] if "date" in df.columns else "N/A")
print("End:", df["date"].iloc[-1] if "date" in df.columns else "N/A")

In [ ]:
# ============================================================
# Historical closing-price visualization
# ============================================================

plt.figure(figsize=(12, 5))
plt.plot(
    df["date"] if "date" in df.columns else np.arange(len(close_prices)),
    close_prices,
)
plt.title("AAPL Historical Closing Price")
plt.xlabel("Date" if "date" in df.columns else "Time Step")
plt.ylabel("Closing Price")
plt.tight_layout()
plt.show()

## Train/test preparation

To avoid using future information while fitting the scaler, the `MinMaxScaler` is fitted **only on the training portion**. The test portion is transformed using that training scaler.

The test sequences retain the final `TIME_STEP` training observations so the first test prediction has the required historical context.

In [ ]:
# ============================================================
# Train/test split and scaling
# ============================================================

training_size = int(len(close_prices) * TRAIN_RATIO)

if training_size <= TIME_STEP:
    raise ValueError("Training set is too small for the selected TIME_STEP.")

train_raw = close_prices[:training_size]
test_raw = close_prices[training_size:]

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_raw)
test_scaled = scaler.transform(test_raw)

# Include the last TIME_STEP training values as context for test sequences.
test_with_context = np.concatenate(
    [train_raw[-TIME_STEP:], test_raw],
    axis=0,
)
test_with_context_scaled = scaler.transform(test_with_context)

print("Training rows:", len(train_raw))
print("Testing rows:", len(test_raw))
print("Scaled training shape:", train_scaled.shape)
print("Scaled testing shape:", test_scaled.shape)

In [ ]:
# ============================================================
# Convert a time series into supervised sequences
# ============================================================

def create_dataset(dataset, time_step):
    data_x, data_y = [], []

    for i in range(len(dataset) - time_step):
        data_x.append(dataset[i : i + time_step, 0])
        data_y.append(dataset[i + time_step, 0])

    return np.asarray(data_x), np.asarray(data_y)


X_train, y_train = create_dataset(train_scaled, TIME_STEP)
X_test, y_test = create_dataset(test_with_context_scaled, TIME_STEP)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

## Stacked LSTM model

The model uses three stacked LSTM layers followed by a single regression output neuron.

In [ ]:
# ============================================================
# Build the stacked LSTM model
# ============================================================

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense

model = Sequential([
    Input(shape=(TIME_STEP, 1)),
    LSTM(50, return_sequences=True),
    LSTM(50, return_sequences=True),
    LSTM(50),
    Dense(1),
], name="stacked_lstm")

model.compile(
    optimizer="adam",
    loss="mean_squared_error",
)

model.summary()

In [ ]:
# ============================================================
# Train
# ============================================================

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    shuffle=False,
)

In [ ]:
# ============================================================
# Training-loss visualization
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.legend()
plt.tight_layout()
plt.show()

## Prediction and RMSE evaluation

Because this is a regression problem, performance is evaluated using **Root Mean Squared Error (RMSE)** rather than classification accuracy.

The predictions and target values are transformed back to the original price scale before RMSE is calculated.

In [ ]:
# ============================================================
# Predictions
# ============================================================

train_predict_scaled = model.predict(X_train, verbose=0)
test_predict_scaled = model.predict(X_test, verbose=0)

train_predict = scaler.inverse_transform(train_predict_scaled)
test_predict = scaler.inverse_transform(test_predict_scaled)

y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

train_rmse = np.sqrt(mean_squared_error(y_train_actual, train_predict))
test_rmse = np.sqrt(mean_squared_error(y_test_actual, test_predict))

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE:  {test_rmse:.4f}")

In [ ]:
# ============================================================
# Plot historical prices vs. model predictions
# ============================================================

train_plot = np.full_like(close_prices, np.nan, dtype=float)
test_plot = np.full_like(close_prices, np.nan, dtype=float)

train_start = TIME_STEP
train_end = train_start + len(train_predict)
train_plot[train_start:train_end] = train_predict.reshape(-1)

test_start = training_size
test_end = test_start + len(test_predict)
test_plot[test_start:test_end] = test_predict.reshape(-1)

x_axis = df["date"] if "date" in df.columns else np.arange(len(close_prices))

plt.figure(figsize=(14, 6))
plt.plot(x_axis, close_prices, label="Actual")
plt.plot(x_axis, train_plot, label="Train Prediction")
plt.plot(x_axis, test_plot, label="Test Prediction")
plt.title("AAPL Actual vs LSTM Predictions")
plt.xlabel("Date" if "date" in df.columns else "Time Step")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()

## 30-step future forecasting

The next 30 values are generated recursively. Each new prediction is appended to the rolling 100-step input sequence and then used to predict the following step.

In [ ]:
# ============================================================
# Recursive future forecast
# ============================================================

full_scaled = scaler.transform(close_prices)

rolling_window = full_scaled[-TIME_STEP:].reshape(1, TIME_STEP, 1)
future_scaled = []

for _ in range(FORECAST_DAYS):
    next_value = model.predict(rolling_window, verbose=0)[0, 0]
    future_scaled.append(next_value)

    next_step = np.array([[[next_value]]], dtype=np.float32)
    rolling_window = np.concatenate(
        [rolling_window[:, 1:, :], next_step],
        axis=1,
    )

future_scaled = np.asarray(future_scaled).reshape(-1, 1)
future_prices = scaler.inverse_transform(future_scaled)

print(f"Generated forecasts: {len(future_prices)} days")
print(future_prices[:5].reshape(-1))

In [ ]:
# ============================================================
# Plot future forecast
# ============================================================

history_window = close_prices[-TIME_STEP:].reshape(-1)

if "date" in df.columns:
    last_date = df["date"].iloc[-1]
    future_dates = pd.bdate_range(
        start=last_date + pd.Timedelta(days=1),
        periods=FORECAST_DAYS,
    )
    history_dates = df["date"].iloc[-TIME_STEP:]

    plt.figure(figsize=(14, 6))
    plt.plot(history_dates, history_window, label="Historical")
    plt.plot(future_dates, future_prices.reshape(-1), label="30-Day Forecast")
    plt.title("AAPL 30-Day LSTM Forecast")
    plt.xlabel("Date")
    plt.ylabel("Closing Price")
    plt.legend()
else:
    history_x = np.arange(TIME_STEP)
    future_x = np.arange(TIME_STEP, TIME_STEP + FORECAST_DAYS)

    plt.figure(figsize=(14, 6))
    plt.plot(history_x, history_window, label="Historical")
    plt.plot(future_x, future_prices.reshape(-1), label="30-Step Forecast")
    plt.title("AAPL Future LSTM Forecast")
    plt.xlabel("Time Step")
    plt.ylabel("Closing Price")
    plt.legend()

plt.tight_layout()
plt.show()

## Notes

- This project is a time-series regression experiment using historical AAPL closing prices.
- The forecast is generated recursively for 30 future time steps.
- RMSE is the primary evaluation metric in this notebook.
- The repository is designed so the notebook can be executed in Google Colab or locally once `data/AAPL.csv` is available.